# Lesson 6A - Modern Python: Searching, Transforming, and Reusing Records

**Goals:** search lists, dictionaries, and sets; convert representations; use zip, enumerate,
unpacking, comprehensions, and lambda; connect functions, objects, dataclasses, type hints,
iterators, and generators in a practical record-processing workflow.

**Before you begin:** complete [Lesson 6: Dictionaries and Sets](06_dictionaries_and_sets.ipynb)
and Lessons 1–5. This notebook introduces its own function/object vocabulary so you can try it
now. It is also a useful review after Lessons 7, 9, and 13–16, which explore these subjects more deeply.
The repository's current Lesson 6 filename contains `and`; it is unchanged.

**Setup:** Python 3.11+ and your existing Jupyter setup. All examples use the standard library;
no new packages, internet, API keys, database, or real employee data are needed.

Run cells in order with **Shift+Enter**. Before each example, predict its output; run it,
change one value, and explain the difference. Restore the inputs and Restart Kernel / Run All
to check independence from old state. All data below is fictional.

Work through the practice cells before opening
`instructor_solutions/06_A_modern_function_solutions.ipynb`. Check values are supplied, but full
exercise implementations are kept in that separate answer key.

## 1. Start with one collection of named records

A list preserves row order; each dictionary gives one row named fields. We will search and
transform the same three fictional employees. Amounts are monthly basic pay, in rupees. Simple
DA examples use the earlier worksheet rate of 48%; they are not actual payroll advice.

Choose the representation for the question: an ordered list for a queue, a dictionary for
lookup by a unique ID, a set for membership, or an object for named state and behavior.

In [ ]:
employees = [
    {"id":101,"name":"Asha","department":"Finance","basic":40_000.},
    {"id":102,"name":"Arun","department":"IT","basic":60_000.},
    {"id":103,"name":"Neha","department":"Finance","basic":50_000.},
]
print(employees[0]["name"], len(employees))
assert len(employees) == 3

## 2. Search lists: membership, first index, and all positions

`value in values` asks whether a value occurs. `.index(value)` returns the first zero-based
position, and raises ValueError when absent; guard it with membership if absence is normal.
`enumerate(values)` supplies a position together with each value. Its optional start changes
the displayed counter, not the underlying list's indexes.

Predict the first Finance index and both matching positions. Change the first department to HR
and explain why membership and first index can tell different stories.

In [ ]:
departments = ["Finance","IT","Finance"]
print("Finance" in departments, "Legal" in departments)
if "Finance" in departments:
    print("First Finance position:", departments.index("Finance"))
positions = []
for position, department in enumerate(departments):
    if department == "Finance":
        positions.append(position)
print("All matching positions:", positions)
for row_number, department in enumerate(departments, start=1):
    print(row_number, department)
assert positions == [0,2]

## 3. Search dictionaries: keys, values, and exact pairs

`in record` searches keys, not values. `.get(key, default)` reads an optional value without
inserting it. `keys()`, `values()`, and `items()` expose keys, values, and `(key,value)` pairs.
Membership in items checks an exact pair. Pair membership also distinguishes an absent key
from a present key holding None. Views reflect later changes; `list(view)` is a snapshot of
the outer sequence, not a deep copy of nested values.

In [ ]:
employee = {"id":101,"name":"Asha","department":"Finance","review_date":None}
print("name" in employee, "Asha" in employee, "Asha" in employee.values())
print(employee.get("manager","Not recorded"))
print(list(employee.keys()))
print(list(employee.values()))
print(list(employee.items()))
print(("department","Finance") in employee.items())
assert ("review_date",None) in employee.items()
assert ("missing",None) not in employee.items()
assert employee.get("review_date") is None and employee.get("missing") is None

## 4. Search a list of dictionaries

A membership check against the outer list compares entire dictionaries. To match one field,
visit each row and compare that field. Use break for the first match; collect all matches when
the field is not unique. The None default makes the not-found case explicit.

The matched dictionary below is the original object, not a copy. Editing it would edit the
source record. Use `.copy()` when you want a separate outer record.

In [ ]:
target_id = 102
found = None
for row in employees:
    if row["id"] == target_id:
        found = row
        break
finance_rows = []
for row in employees:
    if row.get("department") == "Finance":
        finance_rows.append(row)
print(found)
print(finance_rows)
assert found is employees[1]
assert len(finance_rows) == 2

## 5. Search sets and choose a lookup structure

Sets store unique hashable values and offer membership without a positional index. They
have no guaranteed iteration order; sort for predictable display. `required - trained` answers
which required IDs are missing. A set cannot hold dictionaries because dictionaries are mutable
and unhashable; store their IDs instead.

A dictionary/set lookup is typically faster than scanning a large list for each repeated
query (average constant-time membership versus a linear scan). Building a lookup has a cost,
and repeated dictionary keys overwrite records unless you detect duplicates.

In [ ]:
trained = {101,103}
required = {101,102,103}
print(102 in trained)
print("Missing:", sorted(required-trained))
print("Both:", sorted(required & trained))
print("Either:", sorted(required | trained))
assert required-trained == {102}
by_id = {}
for row in employees:
    if row["id"] in by_id:
        print("Duplicate ID:", row["id"])
        continue
    by_id[row["id"]] = row.copy()
assert by_id.get(102)["name"] == "Arun"
assert by_id.get(999) is None

## 6. Convert dictionaries into other structures and back

`list(mapping)` and `tuple(mapping)` contain keys, not values or whole records.
`set(mapping)` makes a unique-key membership set. Use `.values()` or `.items()` to request
values or pairs. `dict(pairs)` constructs a mapping from two-item entries; repeated keys keep
the last value. A set of pairs works only when both parts of every pair are hashable.

Representation changes can lose information: sets discard order/duplicates, and dictionaries
cannot retain repeated keys. Keep IDs when exporting a mapping as records.

In [ ]:
pay_by_id = {101:40_000.,102:60_000.,103:50_000.}
print(list(pay_by_id), tuple(pay_by_id), sorted(set(pay_by_id)))
print(list(pay_by_id.values()))
pairs = list(pay_by_id.items())
assert dict(pairs) == pay_by_id
print(tuple(pay_by_id.items()), sorted(set(pay_by_id.items())))
records = []
for employee_id, basic in pay_by_id.items():
    records.append({"id":employee_id,"basic":basic})
print(records)
assert dict([(101,40_000),(101,45_000)]) == {101:45_000}
assert records[0] == {"id":101,"basic":40_000.}

## 7. Pair parallel data with zip

`zip(a,b)` produces pairs and stops at the shortest input. That default can silently drop
records. `zip(..., strict=True)` raises ValueError if input lengths differ, so it is appropriate
for columns that must align. Validation occurs when zip is consumed, not merely created.
Python 3.11 supports strict=True (introduced in 3.10).

Here try/except catches one expected problem so Run All continues: the indented try block is
attempted; except ValueError runs only for that exception. Errors are not silently ignored.
Even strict zip cannot detect duplicate IDs or columns that are the same length but in the wrong order.

In [ ]:
ids = [101,102,103]
salaries = [40_000.,60_000.,50_000.]
paired_pay = dict(zip(ids,salaries,strict=True))
print(paired_pay)
print("Default zip drops the extra ID:", list(zip(ids,[40_000.])))
try:
    dict(zip(ids,[40_000.],strict=True))
except ValueError:
    print("Rejected: each employee needs exactly one salary.")
assert paired_pay[102] == 60_000.

## 8. Unpack records, sequences, and mappings

Unpacking assigns values from an iterable to several names. A starred target collects
remaining sequence items into a list. `*values` expands a sequence when building a collection
or calling a function. `**mapping` expands named fields; in a dictionary display, later keys win.
Unpacking a dictionary directly visits keys, not its values. Use items() for `(key,value)` pairs.
These operations copy an outer structure only; nested mutable values can remain shared.

In [ ]:
employee_id, employee_name = (101,"Asha")
first, *middle, last = [101,102,103,104]
print(employee_id, employee_name, first, middle, last)
defaults = {"active":True,"department":"Unassigned"}
profile = {**defaults,"department":"Finance","id":101}
all_ids = [*ids,104]
print(profile, all_ids)
key1, key2 = {"id":101,"name":"Asha"}
assert (key1,key2) == ("id","name")
original = {"notes":["Review"]}
copy = {**original}
copy["notes"].append("Approved")
assert original["notes"] == ["Review","Approved"]

## 9. Comprehensions express small transformations

A list comprehension has the form `[expression for item in source if condition]`.
Read it as: visit items, keep those meeting the condition, then produce the expression.
Dictionary comprehensions produce key/value entries; set comprehensions deduplicate results.
Use a normal loop for multiple validation steps or rejection reasons. A compact comprehension
is not automatically clearer, and repeated dictionary keys still overwrite earlier entries.

Predict the names for basic pay at least 50,000. Raise the threshold to 60,000 and explain the result.

In [ ]:
higher_paid = [row["name"] for row in employees if row["basic"] >= 50_000]
names_by_id = {row["id"]:row["name"] for row in employees}
departments_set = {row["department"].casefold() for row in employees}
da_values = [row["basic"]*0.48 for row in employees]
print(higher_paid, names_by_id, sorted(departments_set), da_values)
assert higher_paid == ["Arun","Neha"]
assert sum(da_values) == 72_000
recovered = {row["id"]:row["basic"] for row in records}
assert recovered == pay_by_id

## 10. Define reusable functions before using lambda

`def` defines a function. Parameters name inputs; a call supplies arguments. `return` sends
a value to the caller, whereas print only displays it. A default makes an argument optional;
the bare `*` makes subsequent parameters keyword-only. A docstring describes the contract.
Names assigned inside a function are local. Passing inputs and returning results makes
calculations easier to test than relying on changing global state.

`raise ValueError` rejects input that violates the function contract. We check finite values
so NaN/infinity cannot slip through a nonnegative check.

In [ ]:
import math

def calculate_da(basic, *, rate=0.48):
    """Return monthly DA for finite nonnegative pay and a rate in [0,1]."""
    if isinstance(basic,bool) or not isinstance(basic,(int,float)) or not math.isfinite(basic) or basic < 0:
        raise ValueError("Basic must be a finite nonnegative number")
    if isinstance(rate,bool) or not isinstance(rate,(int,float)) or not math.isfinite(rate) or not 0 <= rate <= 1:
        raise ValueError("Rate must be finite and between 0 and 1")
    allowance = basic * rate  # Local name; not a global running total.
    return allowance

result = calculate_da(60_000,rate=0.48)
print(f"Monthly DA: {result:,.2f}")
assert result == 28_800
def display_da(basic):
    print(calculate_da(basic))  # No explicit return means None.
assert display_da(0) is None

## 11. Defaults, variadic arguments, and call unpacking

Default values are created when def runs. A list default would be reused across calls;
use None and construct a fresh list for each omitted argument. `*args` collects extra positional
arguments into a tuple; `**kwargs` collects extra keyword arguments into a dictionary. In a call,
`*` and `**` expand existing data instead. Prefer explicit parameters for a small fixed contract.
Do not pass the same keyword twice through explicit arguments and ** expansion.

In [ ]:
def add_note(note, notes=None):
    if notes is None:
        notes = []
    notes.append(note)
    return notes

def total_readings(*readings):
    return sum(readings)

def make_metadata(**fields):
    return fields

print(total_readings(10,20,30), total_readings(*[10,20,30]))
print(make_metadata(source="demo",unit="kWh"))
options = {"rate":0.5}
assert calculate_da(60_000,**options) == 30_000
assert add_note("first") == ["first"]
assert add_note("second") == ["second"]

## 12. Lambda and higher-order functions

A lambda is a short function containing one expression. `sorted(..., key=...)` calls a key
function for each item and returns a new list. A tuple key sorts field by field, allowing a
tie-breaker. Use a named def for validation, multiple steps, or logic needing a docstring.
A function that accepts another function is called higher-order; the supplied function can act
as a callback or policy.

Predict the order when sorting by descending basic pay. Make two salaries equal and explain
how the ID tie-breaker works.

In [ ]:
ordered = sorted(employees,key=lambda row:(-row["basic"],row["id"]))
print([row["name"] for row in ordered])
assert [row["id"] for row in ordered] == [102,103,101]

def transform_pay(rows, calculation):
    return [calculation(row["basic"]) for row in rows]

assert transform_pay(employees,calculate_da) == [19_200,28_800,24_000]
assert transform_pay(employees,lambda basic: basic*12) == [480_000,720_000,600_000]

## 13. map and filter compared with comprehensions

`map(function, values)` transforms items; `filter(predicate, values)` keeps items for which
the predicate is true. Both return lazy iterators in Python 3. Use list(...) when you need to
inspect or reuse all results. A comprehension often keeps a simple transformation/filter in
one readable place. map can be clear when a useful named function already exists.
The iterator lifecycle is explored next.

In [ ]:
pay_samples = [40_000,-1,60_000]
valid_via_filter = list(filter(lambda pay: pay >= 0,pay_samples))
valid_via_list = [pay for pay in pay_samples if pay >= 0]
da_via_map = list(map(calculate_da,valid_via_filter))
da_via_list = [calculate_da(pay) for pay in valid_via_list]
print(valid_via_list, da_via_list)
assert valid_via_filter == valid_via_list == [40_000,60_000]
assert da_via_map == da_via_list == [19_200,28_800]

## 14. Iterators and generator expressions

An iterable, such as a list, can supply an iterator. `iter` obtains one; `next` consumes an
item. An exhausted iterator raises StopIteration unless next receives a default. A list can
supply a new iterator, but the same iterator is one-pass.

Parentheses create a generator expression: values are computed when consumed rather than
stored in a new list. `sum`, `any`, `all`, and list can consume it. `any`/`all` can stop early;
any(empty) is False and all(empty) is True, so check nonemptiness separately when required.

In [ ]:
cursor = iter([101,102])
assert next(cursor) == 101
assert next(cursor) == 102
assert next(cursor,"finished") == "finished"
assert list(cursor) == []
da_stream = (calculate_da(row["basic"]) for row in employees)
print(sum(da_stream), sum(da_stream))
assert list(da_stream) == []
assert any(row["department"] == "IT" for row in employees)
assert all(row["basic"] >= 0 for row in employees)
assert any([]) is False and all([]) is True

## 15. Generator functions and first-match searching

A function containing yield returns a generator iterator. Each yield emits one value and
pauses the function until the consumer asks for another. This suits a stream of validated
records without building a second complete list. It does not remove storage already used by
the input list. Errors inside the generator happen when it is consumed.

Combine next(generator, default) with a filtering expression for a concise first match.
Use a normal loop when the search needs several steps or a reason for rejecting a row.

In [ ]:
def da_records(rows):
    for row in rows:
        yield {"id":row["id"],"da":calculate_da(row["basic"])}

stream = da_records(employees)
assert next(stream) == {"id":101,"da":19_200}
assert len(list(stream)) == 2
assert list(stream) == []
match = next((row for row in employees if row["id"] == 103),None)
missing = next((row for row in employees if row["id"] == 999),None)
assert match["name"] == "Neha" and missing is None

## 16. Objects hold named state and methods

A class defines a kind of object. Calling the class constructs an instance; `__init__`
initializes its attributes. An attribute belongs to the object, and a method is a function
accessed through it. `self` is that instance. A class is useful when state and related behavior
belong together; a pure function is often enough for one calculation.

This small object groups an ID/name/basic salary and reuses the tested DA function. It does
not replace validation for every field or build a full payroll system. Broader OOP, composition,
and inheritance are developed in Lesson 13.

In [ ]:
class Employee:
    def __init__(self, employee_id, name, basic):
        calculate_da(basic)  # Validate basic using the existing contract.
        self.employee_id = employee_id
        self.name = name
        self.basic = basic
        self.notes = []  # Per-instance mutable state, not a shared class list.

    def monthly_da(self):
        return calculate_da(self.basic)

asha = Employee(101,"Asha",40_000)
arun = Employee(102,"Arun",60_000)
asha.notes.append("Review")
print(asha.name,asha.monthly_da())
assert asha.monthly_da() == 19_200
assert arun.notes == []

## 17. Type hints and dataclasses

Type hints describe expected values for people and static checking tools. Python does not
enforce them automatically. `list[EmployeeRecord]`, `dict[int,str]`, and `EmployeeRecord | None`
express a list of records, an ID/name mapping, and an optional record.

`@dataclass` is a decorator that generates methods such as initialization, readable repr, and
field equality. `field(default_factory=list)` makes an independent list for every instance.
`__post_init__` adds runtime validation after generated initialization. `asdict` converts the
record into a dictionary. Dataclass methods are ordinary methods like those above; Lesson 16
expands this topic. Numeric floats here demonstrate transformations; use Lesson 12's Decimal
policies for exact accounting.

In [ ]:
from dataclasses import dataclass, field, asdict

@dataclass
class EmployeeRecord:
    employee_id: int
    name: str
    basic: float
    department: str = "Unassigned"
    notes: list[str] = field(default_factory=list)

    def __post_init__(self):
        if type(self.employee_id) is not int or self.employee_id <= 0:
            raise ValueError("Employee ID must be a positive integer")
        if not isinstance(self.name,str) or not self.name.strip():
            raise ValueError("Name must be nonblank text")
        if not isinstance(self.department,str) or not self.department.strip():
            raise ValueError("Department must be nonblank text")
        self.name = self.name.strip()
        self.department = self.department.strip()
        calculate_da(self.basic)

    def monthly_da(self) -> float:
        return calculate_da(self.basic)

def find_employee(rows: list[EmployeeRecord], employee_id: int) -> EmployeeRecord | None:
    return next((row for row in rows if row.employee_id == employee_id),None)

typed_rows = [EmployeeRecord(row["id"],row["name"],row["basic"],row["department"]) for row in employees]
typed_rows[0].notes.append("Reviewed")
assert typed_rows[1].notes == []
assert find_employee(typed_rows,102).name == "Arun"
assert find_employee(typed_rows,999) is None
print(asdict(typed_rows[0]))
# Hints document intent; without explicit validation, this function still accepts text.
def echo_number(value: int) -> int:
    return value
assert echo_number("text") == "text"

## 18. Compose a directory and test its behavior

Composition means one object uses other objects. A directory has employee records; it is
not a subclass of an employee. Keep state in the directory and reuse record methods. A dictionary
supports repeated ID lookups; building it must reject duplicate IDs explicitly.
The comprehension in to_records uses `id`, while the dataclass uses `employee_id`: mapping field
names deliberately matters when converting representations.

Assertions below verify known outputs and boundary cases. In production-facing validation use
raise, not assert: Python's optimized mode can disable assertions. The copied outer index still
references mutable EmployeeRecord objects; it is not an immutable snapshot.

In [ ]:
class EmployeeDirectory:
    def __init__(self, rows: list[EmployeeRecord]):
        self.by_id: dict[int,EmployeeRecord] = {}
        for row in rows:
            if row.employee_id in self.by_id:
                raise ValueError("Duplicate employee ID")
            self.by_id[row.employee_id] = row

    def find(self, employee_id: int) -> EmployeeRecord | None:
        return self.by_id.get(employee_id)

    def to_records(self) -> list[dict]:
        return [{"id":row.employee_id,"name":row.name,"basic":row.basic}
                for row in self.by_id.values()]

directory = EmployeeDirectory(typed_rows)
assert directory.find(102).monthly_da() == 28_800
assert directory.find(999) is None
assert directory.to_records()[0]["id"] == 101
try:
    EmployeeDirectory([typed_rows[0],typed_rows[0]])
except ValueError:
    print("Duplicate IDs rejected instead of silently replacing an employee.")
assert EmployeeDirectory([]).to_records() == []

## 19. Common mistakes and debugging

| Symptom | Diagnose and correct |
| --- | --- |
| `list.index` raises ValueError | Test membership first, or use a guarded first-match search |
| A dictionary search misses a visible name | `in mapping` searches keys; use values or items for the intended question |
| A set changes display order | Sets have no guaranteed iteration order; use sorted for display |
| A dictionary contains fewer records than input | Check duplicate keys before building it |
| Parallel columns silently lose records | Consume zip with strict=True; also verify column order and unique IDs |
| A copy unexpectedly changes the original | Inspect nested mutable values; outer copying is shallow |
| A calculation is None | Return the result instead of only printing it |
| A second sum is zero | The iterator was consumed; create a new one for another pass |
| Type hints did not reject bad input | Hints are not runtime validation; add explicit checks at boundaries |
| Objects share notes unexpectedly | Put mutable state in __init__ or use default_factory for dataclasses |

Reduce a failing case to two records, print type/value or the lookup keys, and inspect the first
traceback line in your code. Catch expected exceptions narrowly; do not hide programming errors.
For any missing-result check use `is None`, so a valid zero or empty collection is not mistaken
for an absent record.

## 20. Practice lab

Complete tasks in order. Use the scaffold below each task, define the required inputs, and add
comments explaining why the chosen structure fits. Check values describe outcomes without giving
full implementations. Try a changed input and a missing/invalid case wherever meaningful.

### Exercise 1. Find first and all list matches

For `["HR","IT","HR","Finance"]`, check HR membership, find its first index,
and collect all zero-based HR positions using enumerate. Search safely for Legal too.

**Check:** HR True; first index 0; positions `[0,2]`; Legal absent without an exception.

In [ ]:
practice_departments = ["HR","IT","HR","Finance"]
# TODO: membership, guarded index, and all positions.

### Exercise 2. Match a dictionary pair and a missing value

Use `{"id":201,"department":"IT","review":None}`. Check for the exact department/IT
pair and compare review with missing manager using both membership and get. Search a set
`{201,203}` for 202.

**Check:** pair True; review present, manager absent; both get calls return None; 202 absent.

In [ ]:
practice_record = {"id":201,"department":"IT","review":None}
practice_trained = {201,203}
# TODO: distinguish key, value, pair, and set membership.

### Exercise 3. Convert a mapping without losing IDs

Convert `{201:10,202:20}` into a key list, value tuple, pair list, and list of
`{"id":...,"kwh":...}` records. Rebuild the original mapping from records.

**Check:** keys `[201,202]`; values `(10,20)`; two records; exact mapping round trip.

In [ ]:
practice_energy = {201:10,202:20}
# TODO: choose keys/values/items explicitly and round-trip records.

### Exercise 4. Pair columns and unpack a record

Use IDs `[201,202]` and pay `[30000,40000]` to build a dictionary with strict zip.
Unpack `(201,"Mira",30000)` into three names. Test a short pay list and explain when the error occurs.

**Check:** ID 202 maps to 40000; unpacked name Mira; mismatched columns raise ValueError on consumption.

In [ ]:
practice_ids = [201,202]
practice_pay = [30_000,40_000]
# TODO: strict pairing, unpacking, and mismatch test.

### Exercise 5. Transform and search records

From rows `{id:1,basic:100,dept:" IT "}`, `{id:2,basic:-1,dept:"HR"}`, and
`{id:3,basic:200,dept:"it"}`, keep nonnegative pay, build ID→DA at 48%, and a normalized
department set. Find ID 3 with next and a default; check missing ID 99.

**Check:** DA `{1:48,3:96}`; departments `{"it"}`; found basic 200; missing None.

In [ ]:
practice_rows = [{"id":1,"basic":100,"dept":" IT "},
                 {"id":2,"basic":-1,"dept":"HR"},
                 {"id":3,"basic":200,"dept":"it"}]
# TODO: list/dict/set comprehensions and first-match search.

### Exercise 6. Return a typed result with a configurable rate

Write `annual_da(basic: float, *, rate: float = 0.48) -> float` with a docstring.
Reuse calculate_da validation, return twelve monthly allowances, and keep printing outside.

**Check:** basic 50000 gives 288000 at 48% and 300000 at 50%; negative/nonfinite pay fails;
zero gives zero. Explain what type hints do and do not enforce.

In [ ]:
def practice_annual_da(basic: float, *, rate: float = 0.48) -> float:
    # TODO: reusable calculation, docstring, and checks.
    pass

### Exercise 7. Sort with a lambda and a tie-breaker

Sort rows `[{id:3,basic:100},{id:1,basic:200},{id:2,basic:200}]` by descending
basic then ascending ID. Keep the original order unchanged. Use transform_pay with a named
function or lambda to obtain annual basic amounts in the sorted order.

**Check:** sorted IDs `[1,2,3]`; annual values `[2400,2400,1200]`; source IDs `[3,1,2]`.

In [ ]:
practice_rows = [{"id":3,"basic":100},{"id":1,"basic":200},{"id":2,"basic":200}]
# TODO: sorted key, callback transform, and original-order check.

### Exercise 8. Create a dataclass and use composition

Create two EmployeeRecord instances: ID 201 Mira basic 30000 and ID 202 Dev basic 40000.
Add a note to only Mira; place both records in EmployeeDirectory. Use a method to look up Dev
and calculate monthly DA; convert Mira with asdict. Test a duplicate ID.

**Check:** Dev DA 19200; Dev notes empty; exported employee_id 201; duplicate directory fails.

In [ ]:
# TODO: create records, independent notes, composed directory, conversion, duplicate test.

### Exercise 9. Write and consume a generator

Write a generator yielding nonnegative values from `[10,-1,0,20]`. Consume one value
with next, then sum the remaining values twice. Create a fresh generator for a complete total.

**Check:** first 10; remaining sum 20; second remaining sum 0; fresh complete sum 30.
Explain which values are stored and why valid zero is retained.

In [ ]:
def practice_nonnegative(values):
    # TODO: yield only valid nonnegative values; numeric inputs assumed for this task.
    pass
# TODO: next, repeated sum, and fresh generator checks.

### Exercise 10. Diagnose duplicate IDs and shallow copies

Two source rows both have ID 1, with basic pay 100 then 200. Show what a dictionary
comprehension does, then write a function that rejects duplicates before returning the lookup.
Also demonstrate that copying `{"notes":["Review"]}` shares the nested list, then make an
independent notes list for a second copy.

**Check:** naive mapping retains basic 200; safe builder raises ValueError; changing independent
notes leaves the source unchanged. Explain why a copying strategy depends on the record schema.

In [ ]:
practice_rows = [{"id":1,"basic":100},{"id":1,"basic":200}]
def practice_build_lookup(rows):
    # TODO: explicit duplicate policy.
    pass
# TODO: demonstrate and fix nested-list sharing for this known schema.

## Chapter project - An employee directory report

Write `build_directory_report(rows, trained_ids)` for input dictionaries using the section 1
schema. Construct validated EmployeeRecord objects, reject duplicate IDs, and compose an
EmployeeDirectory. Do not edit input dictionaries. Reject missing fields with a clear ValueError;
use the dataclass's checks for invalid ID/name/department/basic pay. Reject the whole batch on
invalid data so partial reports are not mistaken for complete ones.

Return a JSON-ready dictionary containing:

- `names_by_id`: string ID keys mapped to names (so JSON round trips preserve key type).
- `ranked_ids`: IDs sorted by descending basic pay, ascending ID for ties.
- `missing_training`: sorted employee IDs absent from trained_ids.
- `da_total`: consume a generator of monthly DA values.
- `records`: ordered by ID, each with id, name, department, basic, and monthly_da.

Use the three demonstration employees and trained IDs `{101,103}`.
**Check:** names 101→Asha, 102→Arun, 103→Neha (keys are strings), ranked IDs `[102,103,101]`,
missing training `[102]`, monthly DA total `72000`. Export with json.dumps and read back;
the report must round-trip unchanged. Empty rows give empty collections and total zero.
Test duplicate IDs, a missing basic field, negative/nonfinite basic, and a blank name.

Explain to a colleague where searching, conversion, functions, OOP, type hints, and lazy
iteration each help. This exercise summarizes the tools; later chapters explore their design
tradeoffs more deeply.

In [ ]:
def build_directory_report(rows, trained_ids):
    # TODO: validate -> objects/directory -> searches/transformations -> JSON-ready report.
    pass
# TODO: known results, empty input, invalid inputs, unchanged source, JSON round trip.

## Reflection and knowledge check

1. Which question needs a first index, all matches, a key lookup, or a membership set?
2. What information is lost by converting pairs to a dictionary or a list to a set?
3. What can strict zip detect, and what can it not detect?
4. When is a named function clearer than lambda or a comprehension?
5. How do parameters, arguments, return values, and local variables relate?
6. When do you want a plain dictionary, a dataclass, or a class with methods?
7. Why do type hints not replace runtime validation?
8. How can laziness save intermediate storage while still causing delayed errors or exhaustion?

Choose one example, change an input, predict the result, and explain the observation aloud.
Restart the kernel and Run All after finishing your exercises.

**Continue:** [Lesson 7](07_functions_and_scope.ipynb) for deeper function practice;
[Lesson 9](09_comprehensions_and_unpacking.ipynb) for transformations;
[Lessons 13](13_classes_and_composition.ipynb), [14](14_iterators_and_generators.ipynb),
[15](15_sorting_and_function_tools.ipynb), and [16](16_dataclasses_and_type_hints.ipynb)
for OOP, generators, function tools, and typed records.